# 0. Complete SafeDrive Colab driver

This is the single canonical notebook. The final question is whether geometry-competent SAC can adapt to simulator-controlled procedural traffic, reduce collisions, and retain traffic-free generalization. The learned policy controls one ego vehicle only (`num_agents=1`, `is_multi_agent=false`). Run exactly two seed-0 pilots and one selected seed-1 confirmation; do not add seeds or variants automatically.

## 1. Constants and paths

In [2]:
import json
import os
from pathlib import Path

REPO_URL = "https://github.com/djdhillxn/safedrive.git"
REPO_DIR = Path("/content/safedrive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/SafeDrive")
RUNS_DIR = REPO_DIR / "runs"
CPU_COUNT = os.cpu_count() or 2
N_ENVS = min(4, max(1, CPU_COUNT - 1))
VEC_ENV = "subproc" if N_ENVS > 1 else "dummy"
print({"CPU_COUNT": CPU_COUNT, "N_ENVS": N_ENVS, "VEC_ENV": VEC_ENV})

{'CPU_COUNT': 12, 'N_ENVS': 4, 'VEC_ENV': 'subproc'}


## 2. Runtime, CPU, RAM, CUDA, and L4 inspection

In [3]:
import platform
import psutil
import torch
print(platform.platform())
print(f"CPU: {CPU_COUNT}; RAM: {psutil.virtual_memory().total / 2**30:.1f} GiB")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true

Linux-6.6.122+-x86_64-with-glibc2.35
CPU: 12; RAM: 53.0 GiB
CUDA: True
NVIDIA L4
NVIDIA L4, 23034 MiB


## 3. Google Drive mount

In [4]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 4. Clone or fast-forward pull repository

In [5]:
if not (REPO_DIR / ".git").exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main
    !git -C {REPO_DIR} merge --ff-only origin/main
%cd /content/safedrive

Cloning into '/content/safedrive'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 44 (delta 4), reused 25 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 493.86 KiB | 10.74 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/safedrive


## 5. Install repository and verify package versions

In [6]:
%pip uninstall -y gym
%pip install --disable-pip-version-check -e .
import gymnasium, metadrive, rich, stable_baselines3, tqdm
print("MetaDrive", getattr(metadrive, "__version__", "pinned source"))
print("SB3", stable_baselines3.__version__, "Gymnasium", gymnasium.__version__)
print("tqdm/rich available", tqdm.__version__, rich.__version__ if hasattr(rich, "__version__") else "yes")

Found existing installation: gym 0.25.2
Uninstalling gym-0.25.2:
  Successfully uninstalled gym-0.25.2
Obtaining file:///content/safedrive
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/metadriverse/metadrive.git (to revision 85e5dadc6c7436d324348f6e3d8f8e680c06b4db) to /tmp/pip-install-qbmm2rj8/metadrive-simulator_58f1f2d136db43ee918ab541445e1c08
  Running command git clone --filter=blob:none --quiet https://github.com/metadriverse/metadrive.git /tmp/pip-install-qbmm2rj8/metadrive-simulator_58f1f2d136db43ee918ab541445e1c08
  Running command git rev-parse -q --verify 'sha^85e5dadc6c7436d324348f6e3d8f8e680c06b4db'
  Running command git fetch -q https://github.com/metadriverse/metadrive.git 85e5dadc6c7436d324348f6e3d8f8e680c06b4db
  Resolved https://github.com/metadriverse/metadrive.git to commit 85e5

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


MetaDrive pinned source
SB3 2.9.0 Gymnasium 1.3.0
tqdm/rich available 4.67.3 yes


## 6. Restore artifacts from Drive

In [7]:
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --local-runs "{RUNS_DIR}" --include-training-artifacts

def optional_latest(name):
    pointer = RUNS_DIR / f"latest_{name}.txt"
    if not pointer.exists():
        return None
    value = Path(pointer.read_text().strip())
    return value if value.is_absolute() else REPO_DIR / value

def run_status(run_dir):
    if run_dir is None:
        return "missing"
    state = run_dir / "curriculum_state.json"
    metadata = run_dir / "run_metadata.json"
    path = state if state.exists() else metadata
    return json.loads(path.read_text()).get("status", "unknown")

def training_action(pointer_name):
    run_dir = optional_latest(pointer_name)
    status = run_status(run_dir)
    if status == "complete":
        return "reuse", run_dir
    if status == "paused":
        return "resume", run_dir
    if status == "failed_gate":
        return "terminal", run_dir
    if status == "missing":
        return "start", None
    raise RuntimeError(f"Refusing ambiguous run status {status!r}: {run_dir}")

Drive source: /content/drive/MyDrive/SafeDrive/runs
Local destination: /content/safedrive/runs
Sync complete: 888 files updated across 32 run directories.
Training artifacts: included (full Colab restore).
Latest IDM: /content/safedrive/runs/20260722_042836_idm_baseline_seed0
Latest PHASE2_EXPERT: /content/safedrive/runs/20260722_155526_expert_baseline_seed0
Latest PHASE2_IDM: /content/safedrive/runs/20260722_155326_idm_baseline_seed0
Latest PPO: /content/safedrive/runs/20260722_012443_ppo_phase1_control_ppo_seed0
Latest PPO_PILOT: /content/safedrive/runs/20260721_230755_ppo_control_pilot_ppo_seed0
Latest SAC: /content/safedrive/runs/20260722_012638_sac_phase1_control_sac_seed0
Latest SAC_PHASE2_CURRICULUM_SEED0: /content/safedrive/runs/20260722_071416_sac_phase2_curriculum_sac_seed0
Latest SAC_PHASE2_CURRICULUM_SEED1: /content/safedrive/runs/20260722_092850_sac_phase2_curriculum_sac_seed1
Latest SAC_PHASE2_CURRICULUM_SEED2: /content/safedrive/runs/20260722_130907_sac_phase2_curriculum

## 7. Compile, test, MetaDrive smoke test, and chase-camera smoke test

In [8]:
!python -m compileall saferl_drive scripts tests
if _exit_code != 0:
    raise RuntimeError("Python compilation failed; stop before running experiments.")
!pytest -q
if _exit_code != 0:
    raise RuntimeError("Repository tests failed; stop before running experiments.")
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split validation --episodes 1 --densities 0.05 --prefix traffic_smoke --no-progress validation.num_scenarios=1
if _exit_code != 0:
    raise RuntimeError("The vector-observation traffic smoke test failed.")
!python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed 50000 --steps 5 --width 320 --height 192 --output /tmp/safedrive_chase_smoke.mp4
if _exit_code != 0:
    chase_log = RUNS_DIR / "video_chase.log"
    if chase_log.exists():
        print(chase_log.read_text()[-5000:])
    raise RuntimeError("The frame-producing chase smoke test failed; diagnostics are printed above.")
assert Path("/tmp/safedrive_chase_smoke.mp4").exists()
print("Compile, tests, traffic smoke, progress dependencies, and frame-producing chase smoke passed.")

Listing 'saferl_drive'...
Compiling 'saferl_drive/algorithms.py'...
Compiling 'saferl_drive/envs.py'...
Compiling 'saferl_drive/evaluation.py'...
Compiling 'saferl_drive/utils.py'...
Listing 'scripts'...
Compiling 'scripts/compare_runs.py'...
Compiling 'scripts/evaluate.py'...
Compiling 'scripts/evaluate_baseline.py'...
Compiling 'scripts/plot_results.py'...
Compiling 'scripts/record_video.py'...
Compiling 'scripts/train.py'...
Compiling 'scripts/train_curriculum.py'...
Listing 'tests'...
Compiling 'tests/test_priority0.py'...
Compiling 'tests/test_sync_drive_runs.py'...
...................................................F                     [100%]
=================================== FAILURES ===================================
_____ test_canonical_notebook_is_valid_concise_and_has_all_twenty_sections _____

    def test_canonical_notebook_is_valid_concise_and_has_all_twenty_sections():
        notebook = json.loads(Path("notebooks/phase2_colab_driver.ipynb").read_text())
        text

AssertionError: 

## 8. Summarize existing Phase-1 and Phase-2 results without retraining

In [ ]:
import pandas as pd
for path in [RUNS_DIR / "phase1_comparison.csv", RUNS_DIR / "phase2_comparison.csv", RUNS_DIR / "phase2_light_traffic_comparison.csv"]:
    if path.exists():
        display(pd.read_csv(path))
print("Historical Phase 1/2 runs are evidence and are never retrained by this notebook.")

## 9. Experiment 0: IDM and ExpertPolicy traffic solvability

In [ ]:
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy idm --split validation --episodes 50 --densities 0.05 0.10 --prefix traffic_solvability_idm validation.num_scenarios=50
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split validation --episodes 50 --densities 0.05 0.10 --prefix traffic_solvability_expert validation.num_scenarios=50
!python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed 50000 --steps 250 --output runs/videos/expert_solubility_chase_d005_seed50000.mp4
for name in ["traffic_extension_idm", "traffic_extension_expert"]:
    baseline_run = optional_latest(name)
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{baseline_run}"

## 10. Experiment 1: frozen pre-adaptation evaluation matrix

In [ ]:
for seed in [0, 1]:
    source_run = optional_latest(f"sac_phase2_curriculum_seed{seed}")
    if source_run is None:
        raise FileNotFoundError(f"Restore the completed curriculum seed-{seed} run from Drive.")
    !python -m scripts.evaluate --run-dir "{source_run}" --model best --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_before
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{source_run}"
SOURCE_SEED0_RUN = optional_latest("sac_phase2_curriculum_seed0")
!python -m scripts.evaluate --run-dir "{SOURCE_SEED0_RUN}" --model best --split validation --episodes 50 --densities 0.0 0.05 0.10 --prefix traffic_source_validation validation.num_scenarios=50
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{SOURCE_SEED0_RUN}"

## 11. Experiment 2: seed-0 SAC-Traffic pilot

In [ ]:
action, REFERENCE_RUN = training_action("sac_traffic_reference_seed0")
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant reference --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant reference --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{REFERENCE_RUN}"
else:
    print(f"Reference pilot action: {action}; run: {REFERENCE_RUN}")
REFERENCE_RUN = optional_latest("sac_traffic_reference_seed0")
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{REFERENCE_RUN}"

## 12. Experiment 3: seed-0 SAC-Traffic-Safe pilot

In [ ]:
action, SAFETY_RUN = training_action("sac_traffic_safety_seed0")
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant safety --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant safety --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{SAFETY_RUN}"
else:
    print(f"Safety pilot action: {action}; run: {SAFETY_RUN}")
SAFETY_RUN = optional_latest("sac_traffic_safety_seed0")
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{SAFETY_RUN}"

## 13. Pilot comparison and saved selection decision

In [ ]:
for pilot_run in [REFERENCE_RUN, SAFETY_RUN]:
    pilot_status = run_status(pilot_run)
    if pilot_status not in {"complete", "failed_gate"}:
        raise RuntimeError(f"Pilot is not terminal: {pilot_status!r} ({pilot_run})")
    if pilot_status == "failed_gate":
        print(f"Evaluating preserved Stage-A failure checkpoint without treating it as completed adaptation: {pilot_run}")
    !python -m scripts.evaluate --run-dir "{pilot_run}" --model best --split validation --episodes 50 --densities 0.0 0.05 0.10 --prefix traffic_pilot validation.num_scenarios=50
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{pilot_run}"
!python -m scripts.compare_runs --traffic-extension --select-pilots
SELECTION = json.loads((RUNS_DIR / "traffic_extension_selection.json").read_text())
print(json.dumps(SELECTION, indent=2))

## 14. Experiment 5: selected seed-1 confirmation

In [ ]:
SELECTED_VARIANT = SELECTION["selected_variant"]
pointer = f"sac_traffic_{SELECTED_VARIANT}_seed1"
action, CONFIRMATION_RUN = training_action(pointer)
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant {SELECTED_VARIANT} --seed 1 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant {SELECTED_VARIANT} --seed 1 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{CONFIRMATION_RUN}"
else:
    print(f"Confirmation action: {action}; run: {CONFIRMATION_RUN}")
CONFIRMATION_RUN = optional_latest(pointer)
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{CONFIRMATION_RUN}"

## 15. Final evaluation matrix

In [ ]:
confirmation_status = run_status(CONFIRMATION_RUN)
if confirmation_status not in {"complete", "failed_gate"}:
    raise RuntimeError(f"The seed-1 confirmation is not terminal: {confirmation_status!r}.")
if confirmation_status == "failed_gate":
    print("Seed 1 failed its Stage-A gate. Its saved best diagnostic checkpoint will be evaluated, but it will be excluded from completed-lineage aggregates.")
SELECTED_SEED0_RUN = optional_latest(f"sac_traffic_{SELECTED_VARIANT}_seed0")
for adapted_run in [SELECTED_SEED0_RUN, CONFIRMATION_RUN]:
    !python -m scripts.evaluate --run-dir "{adapted_run}" --model best --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{adapted_run}"
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy idm --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final_idm
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final_expert
for name in ["traffic_extension_idm", "traffic_extension_expert"]:
    baseline_run = optional_latest(name)
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{baseline_run}"

## 16. Third-person videos

In [ ]:
from scripts.record_video import select_video_scenario
ADAPTED_CSV = SELECTED_SEED0_RUN / "eval" / "traffic_final_d005_episodes.csv"
MATCHED_SEED = select_video_scenario(ADAPTED_CSV, "first")
SOURCE_RUN = optional_latest("sac_phase2_curriculum_seed0")
!python -m scripts.record_video --run-dir "{SOURCE_RUN}" --model best --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/frozen_before_matched_chase.mp4
!python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/adapted_after_matched_chase.mp4
adapted_frame = pd.read_csv(ADAPTED_CSV)
if adapted_frame["success"].astype(bool).any():
    !python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --episodes-csv "{ADAPTED_CSV}" --scenario-rule first_success --output runs/videos/adapted_representative_success_chase.mp4
else:
    print("No successful selected-policy episode exists; no success video will be fabricated.")
if (~adapted_frame["success"].astype(bool)).any():
    !python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --episodes-csv "{ADAPTED_CSV}" --scenario-rule first_failure --output runs/videos/adapted_diagnostic_failure_chase.mp4
!python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/expert_demonstration_chase.mp4

## 17. Final plots and tables

In [ ]:
!python -m scripts.compare_runs --traffic-extension
display(pd.read_csv(RUNS_DIR / "traffic_extension_comparison.csv"))

## 18. Build main and surrogate reports

In [ ]:
!latexmk -cd -pdf -interaction=nonstopmode -halt-on-error reports/main.tex
!latexmk -cd -pdf -interaction=nonstopmode -halt-on-error reports/surrogate_notes.tex

## 19. Final Drive synchronization

In [ ]:
for run_dir in [SELECTED_SEED0_RUN, CONFIRMATION_RUN]:
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{run_dir}"
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --project-artifacts-to-drive
print("Final artifacts persisted to", DRIVE_PROJECT)